In [ ]:
import pandas as pd
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [9]:
data_root = Path("/Users/jliu/workspace/ICL/results/sample/")
task_type_lst = [ "memorization","id_generalization","ood_same_rule","ood_transfer"]

# shot effect in different stages

In [57]:
def load_data(model_type,task_type,filename:str)->pd.DataFrame:
    df = pd.read_parquet(data_root/model_type/task_type/filename)
    print(f"Row num: {df.shape[0]}")
    return df


def plot_shot(df,model_type,task_type):

    # --- Step 1. Group by checkpoint & context_size ---
    grouped = (
        df.groupby(['checkpoint_step', 'context_size'])['num_correct']
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    grouped['sem'] = grouped['std'] / np.sqrt(grouped['count'])

    # --- Step 2. Sort checkpoints & split into 4 groups ---
    unique_ckpts = sorted(grouped['checkpoint_step'].unique())
    num_groups = 4
    ckpt_groups = np.array_split(unique_ckpts, num_groups)
    stage_labels = [f"Stage {i+1}" for i in range(num_groups)]

    # --- Step 3. Use 4 highly distinct colors ---
    stage_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  
    # blue, orange, green, red → very distinguishable

    # --- Step 4. Plot figure ---
    plt.figure(figsize=(12, 7))

    for stage_idx, (stage_ckpts, color) in enumerate(zip(ckpt_groups, stage_colors)):
        stage_df = grouped[grouped['checkpoint_step'].isin(stage_ckpts)]
        
        # Average across checkpoints in this stage
        stage_avg = (
            stage_df.groupby('context_size')
                    .agg(mean=('mean', 'mean'), sem=('sem', 'mean'))
                    .reset_index()
        )
        
        # Plot stage line with SEM error bars
        plt.errorbar(
            stage_avg['context_size'],
            stage_avg['mean'],
            yerr=stage_avg['sem'],
            fmt='o-',
            color=color,
            label=stage_labels[stage_idx],
            capsize=4,
            elinewidth=1.3,
            linewidth=2.3,
            markersize=6,
            alpha=0.95
        )

    # --- Step 5. Beautify plot ---
    plt.xlabel('Context Size (Shots)', fontsize=13)
    plt.ylabel('Mean Num Correct', fontsize=13)
    plt.title(f"{model_type}_{task_type}", fontsize=15)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(title="Training Stage", fontsize=10)
    plt.tight_layout()
    plt.show()

In [58]:
def run_pipeline(model_type,task_type,filename):
    df = load_data(model_type,task_type,filename)
    plot_shot(df,model_type,task_type)
    return df

In [66]:
filename = "icl_performance_ood_clm_try.parquet"
task_type = "ood_same_rule"
model_type = "clm"
df = load_data(model_type,task_type,filename)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/jliu/workspace/ICL/results/sample/clm/ood_same_rule/icl_performance_ood_clm_try.parquet'

In [70]:

filename = "icl_performance.parquet"
task_type = "memorization"
model_type = "clm"
df = load_data(model_type,task_type,filename)

Row num: 30000


In [71]:
set(df['eval_type'])

{'memorization'}

In [68]:
df.columns

Index(['model_variant', 'checkpoint_step', 'model_id', 'config_L', 'config_m',
       'eval_type', 'context_size', 'control_type', 'sequence_id',
       'target_config_L', 'target_config_m', 'accuracy',
       'appears_in_training'],
      dtype='object')